# TMDB Pandas Analysis

This notebook answers the 10 analysis questions using Pandas and the CSV files in the Dataset folder. Each analysis is followed by a short, data-driven insight.

Missing or zero budget and revenue values are treated as unavailable, not as real zero-dollar transactions. They are excluded from profit, ROI, and analyses that compare financial values.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR_CANDIDATES = [Path.cwd() / 'Dataset', Path.cwd().parent / 'Dataset', Path.cwd(), Path.cwd().parent]
DATA_DIR = next((path for path in DATA_DIR_CANDIDATES if (path / 'movies.csv').exists()), None)
if DATA_DIR is None:
    searched = ', '.join(str(path.resolve()) for path in DATA_DIR_CANDIDATES)
    raise FileNotFoundError(f'Could not find movies.csv. Searched: {searched}')

def load_csv(name):
    frame = pd.read_csv(DATA_DIR / f'{name}.csv', keep_default_na=False)
    for column in frame.select_dtypes(include='object').columns:
        frame[column] = frame[column].astype('string').str.strip()
    return frame.drop_duplicates().reset_index(drop=True)

movies = load_csv('movies')
genres = load_csv('genres')
cast = load_csv('cast')
crew = load_csv('crew')
movie_genres = load_csv('movie_genres')
movie_keywords = load_csv('movie_keywords')

for column in ['movie_id', 'runtime', 'budget', 'revenue', 'vote_count']:
    movies[column] = pd.to_numeric(movies[column], errors='coerce')
for column in ['popularity', 'vote_average']:
    movies[column] = pd.to_numeric(movies[column], errors='coerce')
movies['release_date'] = pd.to_datetime(movies['release_date'], errors='coerce')
movies['release_year'] = movies['release_date'].dt.year.astype('Int64')
for column in ['movie_id', 'person_id', 'genre_id']:
    if column in cast:
        cast[column] = pd.to_numeric(cast[column], errors='coerce')
    if column in crew:
        crew[column] = pd.to_numeric(crew[column], errors='coerce')
    if column in genres:
        genres[column] = pd.to_numeric(genres[column], errors='coerce')
for frame in [movie_genres, movie_keywords]:
    for column in ['movie_id', 'genre_id', 'keyword_id']:
        if column in frame:
            frame[column] = pd.to_numeric(frame[column], errors='coerce')

movies['budget_available'] = movies['budget'].notna() & movies['budget'].gt(0)
movies['revenue_available'] = movies['revenue'].notna() & movies['revenue'].gt(0)
print(f'Data directory: {DATA_DIR.resolve()}')
print(f'Loaded {len(movies):,} movies, {len(genres):,} genres, {len(cast):,} cast rows, and {len(crew):,} crew rows.')

Data directory: D:\Guvi\TMDB_Movie_Analytics\Dataset
Loaded 2,503 movies, 19 genres, 24,902 cast rows, and 7,289 crew rows.


C:\Users\nihaa\AppData\Local\Temp\ipykernel_37532\2325522603.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in frame.select_dtypes(include='object').columns:
C:\Users\nihaa\AppData\Local\Temp\ipykernel_37532\2325522603.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/use

In [ ]:
# 1. Unique movies, genres, cast members, and crew members
unique_counts = pd.Series({
    'unique_movies': movies['movie_id'].nunique(),
    'unique_genres': genres['genre_id'].nunique(),
    'unique_cast_members': cast['person_id'].dropna().nunique(),
    'unique_crew_members': crew['person_id'].dropna().nunique(),
}, name='count')
display(unique_counts.to_frame())
print('Insight: The movie, genre, cast, and crew counts establish the scope of the dataset and use IDs to avoid counting repeated relationship rows as new people or entities.')

,count
unique_movies,2503
unique_genres,19
unique_cast_members,11300
unique_crew_members,2848


Insight: The movie, genre, cast, and crew counts establish the scope of the dataset and use IDs to avoid counting repeated relationship rows as new people or entities.


In [ ]:
# 2. Movies with missing or zero budget or revenue
financial_gaps = movies.loc[
    ~movies['budget_available'] | ~movies['revenue_available'],
    ['movie_id', 'title', 'release_year', 'budget', 'revenue', 'budget_available', 'revenue_available']
].sort_values(['release_year', 'title'], na_position='last')
display(financial_gaps)
print(f'Insight: {len(financial_gaps):,} movies have at least one unavailable financial value. Treating zero and missing values as unknown prevents false profits and infinite or misleading ROI values in later analyses.')

,movie_id,title,release_year,budget,revenue,budget_available,revenue_available
2245,653,Nosferatu,1922,0.0,27964.0,False,True
2231,832,M,1931,0.0,35274.0,False,True
2357,147,The 400 Blows,1959,0.0,0.0,False,False
2185,422,8½,1963,0.0,0.0,False,False
2287,797,Persona,1966,0.0,250000.0,False,True
...,...,...,...,...,...,...,...
1335,1010581,My Fault,2023,0.0,0.0,False,False
2134,848326,Rebel Moon - Part One: A Child of Fire,2023,83000000.0,0.0,True,False
1996,1005331,Carry-On,2024,47000000.0,0.0,True,False
1817,359410,Road House,2024,85000000.0,0.0,True,False


Insight: 133 movies have at least one unavailable financial value. Treating zero and missing values as unknown prevents false profits and infinite or misleading ROI values in later analyses.


In [ ]:
# 3. Profit and return on investment for movies with valid finances
valid_finances = movies.loc[movies['budget_available'] & movies['revenue_available']].copy()
valid_finances['profit'] = valid_finances['revenue'] - valid_finances['budget']
valid_finances['roi'] = valid_finances['profit'] / valid_finances['budget']
profit_roi = valid_finances[['movie_id', 'title', 'release_year', 'budget', 'revenue', 'profit', 'roi']].sort_values('profit', ascending=False)
display(profit_roi)
print(f'Insight: Profit is revenue minus budget, while ROI measures profit relative to budget. {len(profit_roi):,} movies have enough information for both metrics.')

,movie_id,title,release_year,budget,revenue,profit,roi
4,19995,Avatar,2009,237000000.0,2.923706e+09,2.686706e+09,11.336312
16,299534,Avengers: Endgame,2019,356000000.0,2.799439e+09,2.443439e+09,6.863593
19,597,Titanic,1997,200000000.0,2.264162e+09,2.064162e+09,10.320812
178,76600,Avatar: The Way of Water,2022,460000000.0,2.334485e+09,1.874485e+09,4.074967
65,140607,Star Wars: The Force Awakens,2015,245000000.0,2.068224e+09,1.823224e+09,7.441729
...,...,...,...,...,...,...,...
752,512195,Red Notice,2021,160000000.0,1.781430e+05,-1.598219e+08,-0.998887
1965,800158,The Killer,2023,175000000.0,3.621130e+05,-1.746379e+08,-0.997931
1263,588228,The Tomorrow War,2021,200000000.0,1.440000e+07,-1.856000e+08,-0.928000
1254,725201,The Gray Man,2022,200000000.0,4.540230e+05,-1.995460e+08,-0.997730


Insight: Profit is revenue minus budget, while ROI measures profit relative to budget. 2,370 movies have enough information for both metrics.


In [ ]:
# 4. Top 15 ROI movies with budgets of at least $1 million
top_roi = (
    valid_finances.loc[valid_finances['budget'].ge(1_000_000), ['movie_id', 'title', 'release_year', 'budget', 'revenue', 'profit', 'roi']]
    .sort_values(['roi', 'profit'], ascending=False)
    .head(15)
)
display(top_roi)
print('Insight: Among movies with budgets of at least $1 million, the highest ROI titles generated the strongest percentage return rather than necessarily the largest absolute profit.')

,movie_id,title,release_year,budget,revenue,profit,roi
1767,503314,Dragon Ball Super: Broly,2018,1000000.0,125002821.0,124002821.0,124.002821
535,408,Snow White and the Seven Dwarfs,1938,1488423.0,184925486.0,183437063.0,123.242561
1791,36685,The Rocky Horror Picture Show,1975,1400000.0,171181400.0,169781400.0,121.272429
467,1366,Rocky,1976,1000000.0,117253345.0,116253345.0,116.253345
1227,770,Gone with the Wind,1939,4000000.0,402352579.0,398352579.0,99.588145
716,9325,The Jungle Book,1967,4000000.0,378000000.0,374000000.0,93.500000
635,11224,Cinderella,1950,2900000.0,263600000.0,260700000.0,89.896552
372,176,Saw,2004,1200000.0,104045735.0,102845735.0,85.704779
719,12230,One Hundred and One Dalmatians,1961,3600000.0,303000000.0,299400000.0,83.166667
263,601,E.T. the Extra-Terrestrial,1982,10500000.0,797307407.0,786807407.0,74.934039


Insight: Among movies with budgets of at least $1 million, the highest ROI titles generated the strongest percentage return rather than necessarily the largest absolute profit.


In [ ]:
# 5. Movies released per year
movies_per_year = (
    movies.dropna(subset=['release_year'])
    .groupby('release_year', as_index=False)['movie_id']
    .nunique()
    .rename(columns={'movie_id': 'movie_count'})
    .sort_values('release_year')
)
peak_year = movies_per_year.loc[movies_per_year['movie_count'].idxmax()]
display(movies_per_year)
print(f'Insight: The highest output year is {int(peak_year["release_year"])}, with {int(peak_year["movie_count"]):,} movies in this dataset.')

,release_year,movie_count
0,1921,1
1,1922,1
2,1927,1
3,1931,2
4,1936,1
...,...,...
84,2022,65
85,2023,62
86,2024,43
87,2025,26


Insight: The highest output year is 2016, with 121 movies in this dataset.


In [ ]:
# 6. Runtime statistical outliers
runtime_data = movies.loc[movies['runtime'].gt(0)].copy()
q1_runtime = runtime_data['runtime'].quantile(0.25)
q3_runtime = runtime_data['runtime'].quantile(0.75)
iqr_runtime = q3_runtime - q1_runtime
lower_runtime = q1_runtime - 1.5 * iqr_runtime
upper_runtime = q3_runtime + 1.5 * iqr_runtime
runtime_outliers = runtime_data.loc[(runtime_data['runtime'] < lower_runtime) | (runtime_data['runtime'] > upper_runtime), ['movie_id', 'title', 'release_year', 'runtime']].sort_values('runtime')
display(runtime_outliers)
print(f'Insight: Using the 1.5 x IQR rule, runtimes below {lower_runtime:.1f} minutes or above {upper_runtime:.1f} minutes are outliers; {len(runtime_outliers):,} movies meet that rule.')

,movie_id,title,release_year,runtime
2307,774752,The Guardians of the Galaxy Holiday Special,2022,45
2447,198375,The Garden of Words,2013,46
514,693134,Dune: Part Two,2024,167
620,3131,Gangs of New York,2002,168
110,857,Saving Private Ryan,1998,169
397,474350,It Chapter Two,2019,169
0,157336,Interstellar,2014,169
146,285,Pirates of the Caribbean: At World's End,2007,169
76,49051,The Hobbit: An Unexpected Journey,2012,169
492,949,Heat,1995,170


Insight: Using the 1.5 x IQR rule, runtimes below 58.5 minutes or above 166.5 minutes are outliers; 60 movies meet that rule.


,movie_id,title,genre_name,popularity
312,19913,(500) Days of Summer,Comedy,12.4817
463,333371,10 Cloverfield Lane,Thriller,10.0069
447,4951,10 Things I Hate About You,Comedy,17.7453
1762,7840,"10,000 BC",Adventure,10.8815
1525,11674,101 Dalmatians,Family,10.5920
...,...,...,...,...
644,381283,mother!,Horror,9.7310
2380,537116,"tick, tick... BOOM!",Drama,4.6024
1043,7451,xXx,Action,16.9661
423,47971,xXx: Return of Xander Cage,Action,19.3866


,genre_name,movie_count,average_popularity
14,Science Fiction,119,21.261836
11,Music,10,18.324340
2,Animation,139,16.811212
7,Family,48,16.710635
0,Action,474,16.385188
16,War,23,15.470317
10,Horror,223,14.851835
1,Adventure,244,14.785510
8,Fantasy,93,13.022920
12,Mystery,29,12.427131


Insight: Primary genre is defined as the first genre relationship listed for each movie; the grouped table shows how average popularity differs across those primary genres.


...

Insight: 49 titles occur more than once; comparing their release years helps distinguish duplicates from remakes or separate movies sharing a title.
Unusually high popularity with low vote count:


,movie_id,title,release_year,popularity,vote_count,disagreement_score
2388,1275779,Disclosure Day,2026,394.3346,2328,22.255484
46,634649,Spider-Man: No Way Home,2021,438.4757,22429,21.013557
1315,1339713,Obsession,2026,364.8722,4203,19.669142
2248,1083381,Backrooms,2026,258.4743,2483,14.648575
713,687163,Project Hail Mary,2026,162.7023,6750,7.735820
62,557,Spider-Man,2002,180.5936,20991,6.880876
35,315635,Spider-Man: Homecoming,2017,169.0857,23456,6.065044
2452,931285,Mortal Kombat II,2026,99.5231,2260,6.024054
1465,936075,Michael,2026,108.7248,3846,5.668778
1707,1226863,The Super Mario Galaxy Movie,2026,97.1210,3343,5.255647


Unusually low popularity with high vote count:


,movie_id,title,release_year,popularity,vote_count,disagreement_score
11,118340,Guardians of the Galaxy,2014,18.9524,29995,-2.624811
34,70160,The Hunger Games,2012,12.2061,23506,-2.601347
101,274,The Silence of the Lambs,1991,7.0584,18235,-2.473145
48,297761,Suicide Squad,2016,13.7141,22256,-2.429310
205,49538,X-Men: First Class,2011,1.2614,13680,-2.326395
70,346364,It,2017,13.4174,20496,-2.311872
58,297762,Wonder Woman,2017,14.9573,21050,-2.270162
5,293660,Deadpool,2016,29.3387,33003,-2.206521
244,771,Home Alone,1990,1.8114,12658,-2.169899
40,1771,Captain America: The First Avenger,2011,19.4135,22950,-2.164468


Insight: The disagreement score compares standardized popularity with standardized log vote count, reducing the effect of extreme vote-count skew. Positive values indicate popularity is unusually high for the vote volume; negative values indicate the reverse.


In [13]:
# 7. Primary genre and average popularity
genre_order = movie_genres.dropna(subset=['movie_id', 'genre_id']).copy()
genre_order['genre_order'] = genre_order.groupby('movie_id').cumcount()
primary_genres = genre_order.loc[genre_order['genre_order'].eq(0), ['movie_id', 'genre_id']].merge(genres[['genre_id', 'genre_name']], on='genre_id', how='left').merge(movies[['movie_id', 'title', 'popularity']], on='movie_id', how='left')
primary_genre_popularity = primary_genres.groupby('genre_name', as_index=False).agg(movie_count=('movie_id', 'nunique'), average_popularity=('popularity', 'mean')).sort_values('average_popularity', ascending=False)
display(primary_genres[['movie_id', 'title', 'genre_name', 'popularity']].sort_values('title'))
display(primary_genre_popularity)
print('Insight: Primary genre is defined as the first genre relationship listed for each movie; the grouped table shows how average popularity differs across those primary genres.')

,movie_id,title,genre_name,popularity
312,19913,(500) Days of Summer,Comedy,12.4817
463,333371,10 Cloverfield Lane,Thriller,10.0069
447,4951,10 Things I Hate About You,Comedy,17.7453
1762,7840,"10,000 BC",Adventure,10.8815
1525,11674,101 Dalmatians,Family,10.5920
...,...,...,...,...
644,381283,mother!,Horror,9.7310
2380,537116,"tick, tick... BOOM!",Drama,4.6024
1043,7451,xXx,Action,16.9661
423,47971,xXx: Return of Xander Cage,Action,19.3866


,genre_name,movie_count,average_popularity
14,Science Fiction,119,21.261836
11,Music,10,18.324340
2,Animation,139,16.811212
7,Family,48,16.710635
0,Action,474,16.385188
16,War,23,15.470317
10,Horror,223,14.851835
1,Adventure,244,14.785510
8,Fantasy,93,13.022920
12,Mystery,29,12.427131


Insight: Primary genre is defined as the first genre relationship listed for each movie; the grouped table shows how average popularity differs across those primary genres.


In [14]:
# 8. Top 10 actors by movie count and average rating
actor_movie_ratings = cast.loc[cast['person_id'].notna(), ['person_id', 'actor_name', 'movie_id']].drop_duplicates(['person_id', 'movie_id']).merge(movies[['movie_id', 'vote_average']], on='movie_id', how='left').groupby(['person_id', 'actor_name'], as_index=False).agg(movie_count=('movie_id', 'nunique'), average_rating=('vote_average', 'mean')).sort_values(['movie_count', 'average_rating'], ascending=False).head(10)
display(actor_movie_ratings)
print('Insight: The most prolific actors are ranked by distinct movies, with average rating providing context about the typical reception of the movies in which they appeared.')

,person_id,actor_name,movie_count,average_rating
679,2231,Samuel L. Jackson,39,6.938692
106,287,Brad Pitt,38,7.257289
127,380,Robert De Niro,38,7.228316
37,85,Johnny Depp,37,6.856973
13,31,Tom Hanks,33,7.289758
1272,5293,Willem Dafoe,32,7.114781
413,1245,Scarlett Johansson,32,7.056344
2674,13240,Mark Wahlberg,32,6.630063
161,500,Tom Cruise,31,7.040613
64,192,Morgan Freeman,31,6.963742


Insight: The most prolific actors are ranked by distinct movies, with average rating providing context about the typical reception of the movies in which they appeared.


In [15]:
# 9. Repeated movie titles and release years
repeated_titles = movies.loc[movies['title'].notna() & movies['title'].ne('Unknown title'), ['title', 'release_year', 'movie_id']].groupby('title', as_index=False).agg(movie_count=('movie_id', 'nunique'), release_years=('release_year', lambda values: sorted(values.dropna().astype(int).unique().tolist()))).loc[lambda frame: frame['movie_count'].gt(1)].sort_values('title')
display(repeated_titles)
print(f'Insight: {len(repeated_titles):,} titles occur more than once; comparing their release years helps distinguish duplicates from remakes or separate movies sharing a title.')

,title,movie_count,release_years
66,A Nightmare on Elm Street,2,"[1984, 2010]"
101,Aladdin,2,"[1992, 2019]"
104,Alice in Wonderland,2,"[1951, 2010]"
240,Beauty and the Beast,2,"[1991, 2017]"
344,Carrie,2,"[1976, 2013]"
365,Charlie's Angels,2,"[2000, 2019]"
380,Cinderella,2,"[1950, 2015]"
455,Date Night,2,"[1995, 2010]"
456,Dawn of the Dead,2,"[1978, 2004]"
539,Dumbo,2,"[1941, 2019]"


Insight: 49 titles occur more than once; comparing their release years helps distinguish duplicates from remakes or separate movies sharing a title.


In [16]:
# 10. Popularity and vote-count disagreements
pattern_data = movies.loc[movies['popularity'].notna() & movies['vote_count'].notna()].copy()
pattern_data['log_vote_count'] = np.log1p(pattern_data['vote_count'].clip(lower=0))
pattern_data['popularity_z'] = (pattern_data['popularity'] - pattern_data['popularity'].mean()) / pattern_data['popularity'].std()
pattern_data['vote_count_z'] = (pattern_data['log_vote_count'] - pattern_data['log_vote_count'].mean()) / pattern_data['log_vote_count'].std()
pattern_data['disagreement_score'] = pattern_data['popularity_z'] - pattern_data['vote_count_z']
high_popularity_low_votes = pattern_data.nlargest(10, 'disagreement_score')
low_popularity_high_votes = pattern_data.nsmallest(10, 'disagreement_score')
columns = ['movie_id', 'title', 'release_year', 'popularity', 'vote_count', 'disagreement_score']
print('Unusually high popularity with low vote count:')
display(high_popularity_low_votes[columns])
print('Unusually low popularity with high vote count:')
display(low_popularity_high_votes[columns])
print('Insight: The disagreement score compares standardized popularity with standardized log vote count, reducing the effect of extreme vote-count skew. Positive values indicate popularity is unusually high for the vote volume; negative values indicate the reverse.')

Unusually high popularity with low vote count:


,movie_id,title,release_year,popularity,vote_count,disagreement_score
2388,1275779,Disclosure Day,2026,394.3346,2328,22.255484
46,634649,Spider-Man: No Way Home,2021,438.4757,22429,21.013557
1315,1339713,Obsession,2026,364.8722,4203,19.669142
2248,1083381,Backrooms,2026,258.4743,2483,14.648575
713,687163,Project Hail Mary,2026,162.7023,6750,7.735820
62,557,Spider-Man,2002,180.5936,20991,6.880876
35,315635,Spider-Man: Homecoming,2017,169.0857,23456,6.065044
2452,931285,Mortal Kombat II,2026,99.5231,2260,6.024054
1465,936075,Michael,2026,108.7248,3846,5.668778
1707,1226863,The Super Mario Galaxy Movie,2026,97.1210,3343,5.255647


Unusually low popularity with high vote count:


,movie_id,title,release_year,popularity,vote_count,disagreement_score
11,118340,Guardians of the Galaxy,2014,18.9524,29995,-2.624811
34,70160,The Hunger Games,2012,12.2061,23506,-2.601347
101,274,The Silence of the Lambs,1991,7.0584,18235,-2.473145
48,297761,Suicide Squad,2016,13.7141,22256,-2.429310
205,49538,X-Men: First Class,2011,1.2614,13680,-2.326395
70,346364,It,2017,13.4174,20496,-2.311872
58,297762,Wonder Woman,2017,14.9573,21050,-2.270162
5,293660,Deadpool,2016,29.3387,33003,-2.206521
244,771,Home Alone,1990,1.8114,12658,-2.169899
40,1771,Captain America: The First Avenger,2011,19.4135,22950,-2.164468


Insight: The disagreement score compares standardized popularity with standardized log vote count, reducing the effect of extreme vote-count skew. Positive values indicate popularity is unusually high for the vote volume; negative values indicate the reverse.
